# Ground Plane Estimation from a Single Image

Detecting the **ground plane** (the drivable / walkable surface) in a monocular
image, without any depth sensor or camera calibration. Built for the *AmigoBot*
robot-navigation project as undergraduate research at IIT Madras.

**Pipeline**

1. **Monocular depth** — estimate a relative depth map from a single RGB image
   using MiDaS.
2. **Superpixels + region graph** — over-segment the image with SLIC, then build
   a Region Adjacency Graph (RAG) and hierarchically merge regions with similar
   boundaries to get coherent surface patches.
3. **Plane fitting** — for each region, lift its pixels into 3D `(row, col, depth)`
   and fit a plane via SVD; the plane normal is the direction of least variance.
4. **Region growing** — seed from the bottom-centre region (assumed ground) and
   run a BFS over the RAG, absorbing neighbouring regions whose plane normal is
   near-parallel to the seed's. The union of accepted regions is the ground mask.

The design idea is to keep *structure* through the pipeline — superpixels
preserve object boundaries, and the region graph lets planarity propagate along
genuinely adjacent surfaces rather than across the raw pixel grid.

## Setup

In [ ]:
pip install timm

In [ ]:
import os
import cv2
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

from skimage import filters, segmentation, color
from skimage.future import graph

## 1. Monocular depth (MiDaS)

Loads a MiDaS model and its input transform **once**, then reuses them for every
image. `get_depth_map` returns a relative inverse-depth map at the input
resolution.

In [ ]:
# DPT_Large  -> highest accuracy, slowest
# DPT_Hybrid -> medium accuracy, medium speed   (used here)
# MiDaS_small-> lowest accuracy, fastest
model_type = "DPT_Hybrid"

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
midas = torch.hub.load("intel-isl/MiDaS", model_type).to(device).eval()

# Load transforms once (not per-frame) so video runs don't reload every call.
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = (midas_transforms.dpt_transform
             if model_type in ("DPT_Large", "DPT_Hybrid")
             else midas_transforms.small_transform)

In [ ]:
def get_depth_map(img):
    """Relative inverse-depth map for a BGR image, at the input resolution."""
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    input_batch = transform(img).to(device)

    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()

    return prediction.cpu().numpy()

## 2. Superpixels and RAG merging

Over-segment with SLIC, build a boundary RAG, and merge hierarchically so that
adjacent regions separated only by weak boundaries collapse into single surface
patches.

In [ ]:
def weight_boundary(graph, src, dst, n):
    """Combined boundary weight/count for the edge to `n` after merging src, dst."""
    default = {"weight": 0.0, "count": 0}

    count_src = graph[src].get(n, default)["count"]
    count_dst = graph[dst].get(n, default)["count"]
    weight_src = graph[src].get(n, default)["weight"]
    weight_dst = graph[dst].get(n, default)["weight"]

    count = count_src + count_dst
    return {
        "count": count,
        "weight": (count_src * weight_src + count_dst * weight_dst) / count,
    }


def merge_boundary(graph, src, dst):
    """No per-merge bookkeeping needed for boundary RAGs."""
    pass

In [ ]:
def rag_thresholding(img, thresh, components):
    """SLIC over-segmentation + hierarchical boundary-RAG merging.

    Returns the RAG, the merged labels, and the original SLIC labels.
    """
    edges = filters.sobel(color.rgb2gray(img))
    labels = segmentation.slic(img, compactness=30, n_segments=components, start_label=1)
    g = graph.rag_boundary(labels, edges)

    merged = graph.merge_hierarchical(
        labels, g, thresh=thresh, rag_copy=False,
        in_place_merge=True,
        merge_func=merge_boundary,
        weight_func=weight_boundary,
    )
    return g, merged, labels

## 3. Ground-plane estimation

Fit a plane per region via SVD, then grow a region from the bottom-centre seed
across the RAG, keeping neighbours whose plane normal is near-parallel to the
seed's.

In [ ]:
def labels2points(labels):
    """Map each label -> list of (row, col) pixels belonging to it."""
    pts = {i: [] for i in range(np.amax(labels) + 1)}
    for i in range(labels.shape[0]):
        for j in range(labels.shape[1]):
            pts[labels[i, j]].append((i, j))
    return pts


def label_node(graph, labels, pointlist):
    """Two-way mapping between merged labels and RAG node ids."""
    label2node, node2label = {}, {}
    for node_id, attrs in graph._node.items():
        c = pointlist[attrs["labels"][0]][0]
        label2node[labels[c]] = node_id
        node2label[node_id] = labels[c]
    return label2node, node2label


def get_3d_plane(label, depth_map, pointlist, N=40):
    """Fit a plane to N sampled pixels of a region; return its unit normal.

    Pixels are lifted to (row, col, depth) and the normal is the last left
    singular vector (direction of least variance) of the centred points.
    """
    points = random.sample(pointlist[label], N)
    points3d = np.array([[i, j, depth_map[i, j]] for (i, j) in points]).T
    left, _, _ = np.linalg.svd(points3d - points3d.mean(axis=1, keepdims=True))
    return left[:, -1]

In [ ]:
def estimate_ground_plane(img, components=1000, merge_thresh=0.04, plane_thresh=0.99):
    """Ground-plane detection for a single BGR image.

    Returns (mask, overlay, depth_map):
      mask     - binary ground mask (uint8)
      overlay  - input image with the ground region outlined + tinted green
      depth_map- scaled relative depth used for plane fitting
    """
    depth_map = get_depth_map(img)
    # Scale depth so the z-axis is comparable to pixel (row, col) magnitudes.
    depth_map = depth_map * (depth_map.shape[0] + depth_map.shape[1]) / (2 * np.max(depth_map))

    g, labels, old_labels = rag_thresholding(img, merge_thresh, components)
    pointlist = labels2points(labels)
    old_pointlist = labels2points(old_labels)
    label2node, node2label = label_node(g, labels, old_pointlist)

    # visited is indexed by (merged) label, so size it to the max label.
    visited = [False] * (np.amax(labels) + 1)
    mask = np.zeros(labels.shape, dtype=np.uint8)

    # Seed = bottom-centre region, assumed to be ground.
    start_label = labels[labels.shape[0] - 1, (labels.shape[1] - 1) // 2]
    ground_plane = get_3d_plane(start_label, depth_map, pointlist)

    queue = [label2node[start_label]]
    visited[start_label] = True

    while queue:
        node = queue.pop(0)
        for next_node in g._adj[node]:
            next_label = node2label[next_node]
            if visited[next_label]:
                continue
            next_plane = get_3d_plane(next_label, depth_map, pointlist)
            # Near-parallel normals -> coplanar orientation -> same surface.
            if abs(np.dot(ground_plane, next_plane)) >= plane_thresh:
                queue.append(next_node)
                visited[next_label] = True
        for pt in pointlist[node2label[node]]:
            mask[pt] = 1

    # Build the visualisation overlay.
    overlay = img.copy()
    contours, _ = cv2.findContours(mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    for contour in contours:
        cv2.drawContours(overlay, contour, -1, (0, 255, 0), thickness=3)
    b, gch, r = cv2.split(overlay)
    gch = cv2.add(gch, 50, dst=gch, mask=mask)
    cv2.merge((b, gch, r), overlay)

    return mask, overlay, depth_map

## 4. Run on an image

In [ ]:
def run_on_image(filename, components=1000, merge_thresh=0.04, plane_thresh=0.99,
                 extn=".jpg", input_dir="./input", output_dir="./output"):
    img = cv2.imread(f"{input_dir}/{filename}{extn}")
    mask, overlay, depth_map = estimate_ground_plane(
        img, components=components, merge_thresh=merge_thresh, plane_thresh=plane_thresh)

    os.makedirs(output_dir, exist_ok=True)
    panels = [
        ("input", cv2.cvtColor(img, cv2.COLOR_BGR2RGB), None),
        ("depth", depth_map, "magma"),
        ("ground plane", cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB), None),
        ("mask", mask, "gray"),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for ax, (title, im, cmap) in zip(axes, panels):
        ax.imshow(im, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    cv2.imwrite(f"{output_dir}/masked_{filename}.jpg", overlay)
    return mask, overlay

In [ ]:
# run_on_image("1", components=2000)

## 5. Run on a video

In [ ]:
def run_on_video(filename, components=1000, merge_thresh=0.04, plane_thresh=0.99,
                 sample_every=5, input_dir="./input", output_dir="./output"):
    vid = cv2.VideoCapture(f"{input_dir}/{filename}.mp4")
    frames, count = [], 0
    while True:
        success, img = vid.read()
        if not success:
            break
        if count % sample_every == 0:
            _, overlay, _ = estimate_ground_plane(
                img, components=components, merge_thresh=merge_thresh, plane_thresh=plane_thresh)
            frames.append(overlay)
        count += 1
    vid.release()

    os.makedirs(output_dir, exist_ok=True)
    h, w, _ = frames[0].shape
    writer = cv2.VideoWriter(
        f"{output_dir}/{filename}.avi", cv2.VideoWriter_fourcc(*"DIVX"), 15, (w, h))
    for frame in frames:
        writer.write(frame)
    writer.release()
    print(f"Wrote {len(frames)} frames to {output_dir}/{filename}.avi")

In [ ]:
# run_on_video("5", components=2000)